In [1]:
import pandas as pd
import datetime
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import python_ss.python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials


import os
from decimal import Decimal
import calendar
#importlib.reload(utils)
print(os.getcwd())


z:\Users\suehara\Documents\python\analysis\yojitu


In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import ast
import json
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes
import pandas as pd
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],
)


z:\Users\suehara\Documents\python\analysis\.venv\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 200


In [4]:
# #ロンザンのマスタデータのパスの設定
# path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

# #マスタデータを読み込み
# # master1= 日付・月・カレンダー週・Qデータ(2019/10/1	23-10月	9月5W(23日～1日)	23-1Q)
# master1 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[0,1,2,3])

# # master2= 在籍Q・人マスタ・略・user_id・所属フラグ・ロンザン所属フラグ(23-2Q	五十嵐奏子	五十嵐　igarashi ミドル	0)
# master2 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[6,7,8,9,10,11,12])

# # master3= 人マスタ・略・チーム・レイヤー(大仲研司	大仲	1課	部責)
# master3 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[14,15,16,17])

# # master4= ヨミ表選択・丸め(人事部	人事部紹介)
# master4 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[20,21])

# # master5= 計上Q・修正後ポイント・Q計上時ポイント・掛け率(19-3Q	5,712	7,297	78%)
# master5 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[23,24,25,26])

In [5]:
#転機IDの10000以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '11scU7ixGvt2JYSBQlHkGMKZSLSzYbrmG221CXUGDqDU'
Sheet_NAME = 'masta!'
Sheet_row = "A:D"
RANGE_NAME = Sheet_NAME+Sheet_row
master1 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "G:M"
RANGE_NAME = Sheet_NAME+Sheet_row
master2 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "O:P"
RANGE_NAME = Sheet_NAME+Sheet_row
master4 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "R:U"
RANGE_NAME = Sheet_NAME+Sheet_row
master5 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


In [6]:
# master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
# master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

# master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

# master4 = master4.dropna(subset=['ヨミ表選択'])

# master5 = master5.rename(columns={"計上Q.1": "計上Q","掛け率.1":"掛け率"}) #カラム名変更
# master5 = master5.dropna(subset=['計上Q'])

In [7]:
master1['日付'] = pd.to_datetime(master1['日付']) 

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master4 = master4.dropna(subset=['ヨミ表選択'])

master5 = master5.rename(columns={"計上Q.1": "計上Q","掛け率.1":"掛け率"}) #カラム名変更
master5 = master5.dropna(subset=['計上Q'])

In [8]:
master2.head(10)

,Q,人マスタ,sei_plus,user_id,所属フラグ,ロンザン所属フラグ,レイヤー
1,19-3Q,大仲研司,大仲研,oonaka,ミドル,1,部責
2,19-3Q,角田隆一朗,角田隆,kakuta,ミドル,1,課責リーダー
3,19-3Q,服部圭佑,服部圭,k-hattori,ミドル,1,既存
4,19-3Q,大塚洋平,大塚洋,otsuka,ミドル,1,既存
5,19-3Q,大矢裕士,大矢裕,y-ooya,ミドル,1,既存
6,19-3Q,長崎文昭,長崎文,nagasaki,ミドル,1,課責リーダー
7,19-3Q,阿曽祐介,阿曽祐,aso,ミドル,1,中途
8,19-3Q,宮崎佳彦,宮崎佳,y-miyazaki,ミドル,1,中途
9,19-3Q,隆郁也,隆郁,takashi,ミドル,1,既存
10,19-3Q,鈴木博巳,鈴木博,h-suzuki,ミドル,0,中途


In [9]:
# #日付のマスタデータのパスの設定
# path2 = r"\\172.16.0.232\CoffeeCrazy\総合市場開発部\50　個人フォルダ\40　【大阪】\塩澤\マスタ"
# Q_master = pd.read_excel(path2 + "\Qマスタ.xlsx",usecols=[0,1,2,3,5,8,11,12,13])
# Q_master["日付"] = pd.to_datetime(Q_master["日付"]) #日付データを変換
# Q_master.columns

In [10]:
Sheet_NAME = 'Q営業日!'
Sheet_row = "A:O"
RANGE_NAME = Sheet_NAME+Sheet_row
Q_master = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

Q_master['月'] = pd.to_datetime(Q_master['月'], errors='coerce')

Q_master['日付'] = pd.to_datetime(Q_master['日付'])

In [11]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


#社員データ抽出
sql="""
select
user_id ,
sei_plus,
concat(sei,mei) as seimei
FROM `r-group-bigdata.live_company.syain`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
syain_data = client.query(sql).result().to_dataframe()
syain_data.sample(30)

z:\Users\suehara\Documents\python\analysis\.venv\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,user_id,sei_plus,seimei
209,to-yamaguchi,山口大,山口大輝
2601,takeshi-yamaguchi,None,山口武志
1899,to-yoshioka,吉岡稔,吉岡稔騎
4206,h-konishi,小西遥,小西遥稀
2320,r-nishikawa,西川立,西川立育
4051,kurobe,黒部智,黒部智將
1029,ehara,江原し,江原駿
3921,r-kumamoto,隈本理,隈本理恩
718,d-tsukamoto,塚元大,塚元大介
1834,min-okada,岡田美,岡田美紀


In [12]:
# #業務委託の方など、live_company_syain に載ってない人分はこちらにてデータを追加
# d={'user_id': ['yuko-kyotani','aya-takeshita'],
#   'sei_plus': ['yuko-kyotani','aya-takeshita'],
#   'seimei':['yuko-kyotani','aya-takeshita']}
# ronzan_syain_data = pd.DataFrame(d) 

# syain_data = pd.concat([ronzan_syain_data,syain_data])
# syain_data.head()

In [13]:
# conn3 = pymysql.connect(
#                     host="192.168.5.124",
#                     user="eigyou_kikaku",
#                     password="As6hV2K!k",
#                     db="eigyou_kikaku",
#                     port=3306,
#                     charset='utf8mb4',
#                     cursorclass=pymysql.cursors.DictCursor)


In [14]:
# # 今Q取得
# Q = [i['Q'] for i in get_data("select `Q` from eigyoubi_master where `日付` = curdate() - interval 7 day", conn3)][0]

# # 営業日情報取得
# first_date = [i['日付'] for i in get_data("select `日付` from eigyoubi_master where Q = '{}' order by `日付` asc".format(Q), conn3)][0].strftime('%Y-%m-%d')
# end_date = [i['日付'] for i in get_data("select `日付` from eigyoubi_master where Q = '{}' order by `日付` desc".format(Q), conn3)][0].strftime('%Y-%m-%d')

In [15]:
# print(first_date)
# print(end_date)
# print(Q)

In [16]:
# #日付データを取る
# SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
#           'https://www.googleapis.com/auth/spreadsheets']
# json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
# service = ps.get_auth(SCOPES,json_path)
# SPREADSHEET_ID = '1Gpbg3cMCFGNt4dJ0xfVZJmJfV_l_B9V6IM2QKDK8qoc'
# Sheet_NAME = 'Q営業日!A'
# Sheet_row = ":G"
# RANGE_NAME = Sheet_NAME+Sheet_row
# DATE_DATA = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# DATE_DATA["日付"] = pd.to_datetime(DATE_DATA["日付"]) #日付データを変換
# DATE_DATA = DATE_DATA[["日付","営業日","Q","同営業日比較","初中最終月"]]


# 新規初期交渉設定数

In [17]:

#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

sql = """
with ALLBASE as (
with base3 as (
with base2 as (
with base as (
SELECT
 shoki.id,
 shoki.kohosha_id,
 kosho_setteibi,
 kosho_yoteibi,
 case when syi1.sei_plus is null then shoki.mendan_tanto
       else syi1.sei_plus end as mendan_tanto,
 kosho_jisshibi,
 koho.seimei as kohosha_seimei,
 kohosha_sql.name as kohosha_rank,
 kosho_seq as kaisu,
 case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as kohosha_apsource,
 shokai_sql.juryosha,
 shoki.ap_kakutoku as shokikosho_apkakutokusha_moto,
 case
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is null then shoki.mendan_tanto
    when shoki.ap_kakutoku is null and consts.name != "パートナー紹介" then shoki.mendan_tanto
    else shoki.ap_kakutoku end as shokikosho_apkakutokusha,
   
 case when shoki.partner_id is not null then shoki.partner_id
    when shoki.jinjibu_id is not null then shoki.jinjibu_id
    else null end as juryo_id,

 DATE_DIFF(kosho_jisshibi, LAG(kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY kosho_jisshibi), DAY) AS keikabi
from `r-group-bigdata.live_rhs.shokikoshos` as shoki
left join `r-group-bigdata.live_company.syain` syi1 on shoki.mendan_tanto = syi1.user_id
left join `r-group-bigdata.live_company.syain` syi2 on shoki.ap_kakutoku = syi2.user_id
left join
 (select id,seimei,kohosha_rank from `r-group-bigdata.live_rhs.kohoshas`) as koho ON shoki.kohosha_id = koho.id
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON koho.kohosha_rank = kohosha_sql.code
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as consts ON shoki.ap_source = consts.code
left join
 (select kohosha_id,juryosha from `r-group-bigdata.live_rhs.shokaijuryos`) as shokai_sql ON shoki.kohosha_id = shokai_sql.kohosha_id)

select *,
    LAG(kohosha_apsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_kohosha_apsource,
  from base)

select *,
   CASE WHEN kohosha_apsource != prev_kohosha_apsource THEN 1 ELSE 0
        END AS APS_change
  from base2),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

select
 id,
 kohosha_id,
 kohosha_seimei,
 kosho_setteibi,
 kosho_yoteibi,
 kosho_jisshibi,
 mendan_tanto,
 kohosha_rank,
 kaisu,
 kohosha_apsource,
 juryosha,
 shokikosho_apkakutokusha_moto,
 shokikosho_apkakutokusha,
 juryo_id,
 keikabi,
 prev_kohosha_apsource,
 APS_change,
 keikabi,
 case when kaisu = 1 then 1
      when APS_change = 1 then 1
      when keikabi > 90 then 2
      else 0 end as sai_flg,
 cal.quarter as shoki_setteiQ
from base3
left join cal on base3.kosho_setteibi = cal.date
order by kosho_setteibi desc)

-- 新規初期交渉設定
SELECT 
  "shoki_settei_shin" as type,
  concat("shoki_settei_shin",mendan_tanto) as key,
  mendan_tanto,
  shoki_setteiQ,
  COUNT(shoki_setteiQ) AS count_shoki_setteiQ
FROM ALLBASE
WHERE sai_flg = 1
GROUP BY mendan_tanto, shoki_setteiQ
ORDER BY shoki_setteiQ


"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()
shoki_settei_shin = shokikosho_data.copy()

In [18]:
shoki_settei_shin['join_key'] = shoki_settei_shin['mendan_tanto'] + shoki_settei_shin['shoki_setteiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
shoki_settei_shin = pd.merge(shoki_settei_shin, master2, on='join_key', how='inner')
shoki_settei_shin_kojin = shoki_settei_shin.pivot_table(index=["key","mendan_tanto"],columns="shoki_setteiQ",aggfunc="sum",values="count_shoki_setteiQ").fillna(0)


In [19]:
#  異動や退職を「ー」に変更
def convert_to_dash(x):
    if x == 0:
        return "ー"
    return x

shoki_settei_shin_kojin = shoki_settei_shin_kojin.applymap(convert_to_dash)


C:\Users\suehara\AppData\Local\Temp\ipykernel_23240\2718775389.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  shoki_settei_shin_kojin = shoki_settei_shin_kojin.applymap(convert_to_dash)


In [20]:
shoki_settei_shin["key2"] = shoki_settei_shin["type"] + shoki_settei_shin["レイヤー"]
shoki_settei_shin_layer = shoki_settei_shin.pivot_table(index=["key2","レイヤー"],columns="shoki_setteiQ",aggfunc="sum",values="count_shoki_setteiQ").fillna(0)


In [21]:
shoki_settei_shin

,type,key,mendan_tanto,shoki_setteiQ,count_shoki_setteiQ,join_key,Q,人マスタ,sei_plus,user_id,所属フラグ,ロンザン所属フラグ,レイヤー,key2
0,shoki_settei_shin,shoki_settei_shin大塚洋,大塚洋,19-3Q,38,大塚洋19-3Q,19-3Q,大塚洋平,大塚洋,otsuka,ミドル,1,既存,shoki_settei_shin既存
1,shoki_settei_shin,shoki_settei_shin長崎文,長崎文,19-3Q,33,長崎文19-3Q,19-3Q,長崎文昭,長崎文,nagasaki,ミドル,1,課責リーダー,shoki_settei_shin課責リーダー
2,shoki_settei_shin,shoki_settei_shin大矢裕,大矢裕,19-3Q,23,大矢裕19-3Q,19-3Q,大矢裕士,大矢裕,y-ooya,ミドル,1,既存,shoki_settei_shin既存
3,shoki_settei_shin,shoki_settei_shin角田隆,角田隆,19-3Q,16,角田隆19-3Q,19-3Q,角田隆一朗,角田隆,kakuta,ミドル,1,課責リーダー,shoki_settei_shin課責リーダー
4,shoki_settei_shin,shoki_settei_shin阿曽祐,阿曽祐,19-3Q,7,阿曽祐19-3Q,19-3Q,阿曽祐介,阿曽祐,aso,ミドル,1,中途,shoki_settei_shin中途
5,shoki_settei_shin,shoki_settei_shin服部圭,服部圭,19-3Q,33,服部圭19-3Q,19-3Q,服部圭佑,服部圭,k-hattori,ミドル,1,既存,shoki_settei_shin既存
6,shoki_settei_shin,shoki_settei_shin宮崎佳,宮崎佳,19-3Q,38,宮崎佳19-3Q,19-3Q,宮崎佳彦,宮崎佳,y-miyazaki,ミドル,1,中途,shoki_settei_shin中途
7,shoki_settei_shin,shoki_settei_shin大仲研,大仲研,19-3Q,21,大仲研19-3Q,19-3Q,大仲研司,大仲研,oonaka,ミドル,1,部責,shoki_settei_shin部責
8,shoki_settei_shin,shoki_settei_shin隆郁,隆郁,19-3Q,3,隆郁19-3Q,19-3Q,隆郁也,隆郁,takashi,ミドル,1,既存,shoki_settei_shin既存
9,shoki_settei_shin,shoki_settei_shin大塚洋,大塚洋,19-4Q,17,大塚洋19-4Q,19-4Q,大塚洋平,大塚洋,otsuka,ミドル,1,既存,shoki_settei_shin既存


In [22]:
shoki_settei_shin = pd.concat([shoki_settei_shin_kojin,shoki_settei_shin_layer])

In [23]:
shoki_settei_shin.head(10)

,shoki_setteiQ,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q
shoki_settei_shinay-murakami,ay-murakami,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,21,30,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinaya-takeshita,aya-takeshita,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,192,66,86,101,85,71,73,60,65,67,8
shoki_settei_shinc-yamazaki,c-yamazaki,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,20,44,14,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinfukai,fukai,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,25,67,30,56,2,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinh-matsuda,h-matsuda,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,13,19,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shink-shimoda,k-shimoda,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,14,57,24,45,1,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinkitazawa,kitazawa,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,18,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinm-kubo,m-kubo,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,18,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinma-ishii,ma-ishii,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,20,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_settei_shinma-yoshida,ma-yoshida,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,19,14,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー


# 新規初期交渉実施数

In [24]:
sql = """
with ALLBASE as (
with base3 as (
with base2 as (
with base as (
SELECT
 shoki.id,
 shoki.kohosha_id,
 kosho_setteibi,
 kosho_yoteibi,
 case when syi1.sei_plus is null then shoki.mendan_tanto
       else syi1.sei_plus end as mendan_tanto,
 kosho_jisshibi,
 koho.seimei as kohosha_seimei,
 kohosha_sql.name as kohosha_rank,
 kosho_seq as kaisu,
 case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as kohosha_apsource,
 shokai_sql.juryosha,
 shoki.ap_kakutoku as shokikosho_apkakutokusha_moto,
 case
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is null then shoki.mendan_tanto
    when shoki.ap_kakutoku is null and consts.name != "パートナー紹介" then shoki.mendan_tanto
    else shoki.ap_kakutoku end as shokikosho_apkakutokusha,
   
 case when shoki.partner_id is not null then shoki.partner_id
    when shoki.jinjibu_id is not null then shoki.jinjibu_id
    else null end as juryo_id,

 DATE_DIFF(kosho_jisshibi, LAG(kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY kosho_jisshibi), DAY) AS keikabi
from `r-group-bigdata.live_rhs.shokikoshos` as shoki
left join `r-group-bigdata.live_company.syain` syi1 on shoki.mendan_tanto = syi1.user_id
left join `r-group-bigdata.live_company.syain` syi2 on shoki.ap_kakutoku = syi2.user_id
left join
 (select id,seimei,kohosha_rank from `r-group-bigdata.live_rhs.kohoshas`) as koho ON shoki.kohosha_id = koho.id
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON koho.kohosha_rank = kohosha_sql.code
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as consts ON shoki.ap_source = consts.code
left join
 (select kohosha_id,juryosha from `r-group-bigdata.live_rhs.shokaijuryos`) as shokai_sql ON shoki.kohosha_id = shokai_sql.kohosha_id)

select *,
    LAG(kohosha_apsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_kohosha_apsource,
  from base)

select *,
   CASE WHEN kohosha_apsource != prev_kohosha_apsource THEN 1 ELSE 0
        END AS APS_change
  from base2),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

select
 id,
 kohosha_id,
 kohosha_seimei,
 kosho_setteibi,
 kosho_yoteibi,
 kosho_jisshibi,
 mendan_tanto,
 kohosha_rank,
 kaisu,
 kohosha_apsource,
 juryosha,
 shokikosho_apkakutokusha_moto,
 shokikosho_apkakutokusha,
 juryo_id,
 keikabi,
 prev_kohosha_apsource,
 APS_change,
 keikabi,
 case when kaisu = 1 then 1
      when APS_change = 1 then 1
      when keikabi > 90 then 2
      else 0 end as sai_flg,
 cal.quarter as shoki_jisshiQ
from base3
left join cal on base3.kosho_jisshibi = cal.date
order by kosho_setteibi desc)

-- 新規初期実施設定
SELECT 
  "shoki_jisshi_shin" as type,
  concat("shoki_jisshi_shin",mendan_tanto) as key,
  mendan_tanto,
  shoki_jisshiQ,
  COUNT(shoki_jisshiQ) AS count_shoki_jisshiQ
FROM ALLBASE
WHERE sai_flg = 1
and shoki_jisshiQ is not null
GROUP BY mendan_tanto, shoki_jisshiQ
ORDER BY shoki_jisshiQ


"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()




In [25]:
shoki_jisshi_shin = shokikosho_data.copy()

shoki_jisshi_shin['join_key'] = shoki_jisshi_shin['mendan_tanto'] + shoki_jisshi_shin['shoki_jisshiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
shoki_jisshi_shin = pd.merge(shoki_jisshi_shin, master2, on='join_key', how='inner')
shoki_jisshi_shin_kojin = shoki_jisshi_shin.pivot_table(index=["key","mendan_tanto"],columns="shoki_jisshiQ",aggfunc="sum",values="count_shoki_jisshiQ").fillna(0)

In [26]:
shoki_jisshi_shin["key2"] = shoki_jisshi_shin["type"] + shoki_jisshi_shin["レイヤー"]
shoki_jisshi_shin_layer = shoki_jisshi_shin.pivot_table(index=["key2","レイヤー"],columns="shoki_jisshiQ",aggfunc="sum",values="count_shoki_jisshiQ").fillna(0)


In [27]:
shoki_jisshi_shin = pd.concat([shoki_jisshi_shin_kojin,shoki_jisshi_shin_layer])

In [28]:
shoki_jisshi_shin = shoki_jisshi_shin.applymap(convert_to_dash)


C:\Users\suehara\AppData\Local\Temp\ipykernel_23240\475329382.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  shoki_jisshi_shin = shoki_jisshi_shin.applymap(convert_to_dash)


In [29]:
shoki_jisshi_shin.head(10)

,shoki_jisshiQ,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q
shoki_jisshi_shinay-murakami,ay-murakami,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,19,29,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinaya-takeshita,aya-takeshita,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,162,59,82,92,81,69,71,59,61,59,3
shoki_jisshi_shinc-yamazaki,c-yamazaki,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,14,46,14,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinfukai,fukai,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,20,55,32,49,4,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinh-matsuda,h-matsuda,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,10,21,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shink-shimoda,k-shimoda,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,10,56,22,41,5,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinkitazawa,kitazawa,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,16,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinm-kubo,m-kubo,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,14,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinma-ishii,ma-ishii,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,18,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー
shoki_jisshi_shinma-yoshida,ma-yoshida,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,13,17,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー,ー


# 再交渉設定

In [30]:
sql = """
with ALLBASE as (
with base3 as (
with base2 as (
with base as (
SELECT
 shoki.id,
 shoki.kohosha_id,
 kosho_setteibi,
 kosho_yoteibi,
 case when syi1.sei_plus is null then shoki.mendan_tanto
       else syi1.sei_plus end as mendan_tanto,
 kosho_jisshibi,
 koho.seimei as kohosha_seimei,
 kohosha_sql.name as kohosha_rank,
 kosho_seq as kaisu,
 case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as kohosha_apsource,
 shokai_sql.juryosha,
 shoki.ap_kakutoku as shokikosho_apkakutokusha_moto,
 case
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is null then shoki.mendan_tanto
    when shoki.ap_kakutoku is null and consts.name != "パートナー紹介" then shoki.mendan_tanto
    else shoki.ap_kakutoku end as shokikosho_apkakutokusha,
   
 case when shoki.partner_id is not null then shoki.partner_id
    when shoki.jinjibu_id is not null then shoki.jinjibu_id
    else null end as juryo_id,

 DATE_DIFF(kosho_jisshibi, LAG(kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY kosho_jisshibi), DAY) AS keikabi
from `r-group-bigdata.live_rhs.shokikoshos` as shoki
left join `r-group-bigdata.live_company.syain` syi1 on shoki.mendan_tanto = syi1.user_id
left join `r-group-bigdata.live_company.syain` syi2 on shoki.ap_kakutoku = syi2.user_id
left join
 (select id,seimei,kohosha_rank from `r-group-bigdata.live_rhs.kohoshas`) as koho ON shoki.kohosha_id = koho.id
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON koho.kohosha_rank = kohosha_sql.code
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as consts ON shoki.ap_source = consts.code
left join
 (select kohosha_id,juryosha from `r-group-bigdata.live_rhs.shokaijuryos`) as shokai_sql ON shoki.kohosha_id = shokai_sql.kohosha_id)

select *,
    LAG(kohosha_apsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_kohosha_apsource,
  from base)

select *,
   CASE WHEN kohosha_apsource != prev_kohosha_apsource THEN 1 ELSE 0
        END AS APS_change
  from base2),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

select
 id,
 kohosha_id,
 kohosha_seimei,
 kosho_setteibi,
 kosho_yoteibi,
 kosho_jisshibi,
 mendan_tanto,
 kohosha_rank,
 kaisu,
 kohosha_apsource,
 juryosha,
 shokikosho_apkakutokusha_moto,
 shokikosho_apkakutokusha,
 juryo_id,
 keikabi,
 prev_kohosha_apsource,
 APS_change,
 keikabi,
 case when kaisu = 1 then 1
      when APS_change = 1 then 1
      when keikabi > 90 then 2
      else 0 end as sai_flg,
 cal.quarter as shoki_setteiQ
from base3
left join cal on base3.kosho_setteibi = cal.date
order by kosho_setteibi desc)

-- 新規初期交渉設定
SELECT 
  "shoki_settei_sai" as type,
  concat("shoki_settei_sai",mendan_tanto) as key,
  mendan_tanto,
  shoki_setteiQ as shoki_settei_saiQ,
  COUNT(shoki_setteiQ) AS count_shoki_settei_saiQ
FROM ALLBASE
WHERE sai_flg = 2
GROUP BY mendan_tanto, shoki_setteiQ
ORDER BY shoki_setteiQ


"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
saikosho_data = client.query(sql).result().to_dataframe()


In [31]:
shoki_settei_sai = saikosho_data.copy()

shoki_settei_sai['join_key'] = shoki_settei_sai['mendan_tanto'] + shoki_settei_sai['shoki_settei_saiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
shoki_settei_sai = pd.merge(shoki_settei_sai, master2, on='join_key', how='inner')
shoki_settei_sai_kojin = shoki_settei_sai.pivot_table(index=["key","mendan_tanto"],columns="shoki_settei_saiQ",aggfunc="sum",values="count_shoki_settei_saiQ").fillna(0)




In [32]:
shoki_settei_sai["key2"] = shoki_settei_sai["type"] + shoki_settei_sai["レイヤー"]
shoki_settei_sai_layer = shoki_settei_sai.pivot_table(index=["key2","レイヤー"],columns="shoki_settei_saiQ",aggfunc="sum",values="count_shoki_settei_saiQ").fillna(0)


In [33]:
shoki_settei_sai = pd.concat([shoki_settei_sai_kojin,shoki_settei_sai_layer])

# 再交渉実施数

In [34]:
sql = """
with ALLBASE as (
with base3 as (
with base2 as (
with base as (
SELECT
 shoki.id,
 shoki.kohosha_id,
 kosho_setteibi,
 kosho_yoteibi,
 case when syi1.sei_plus is null then shoki.mendan_tanto
       else syi1.sei_plus end as mendan_tanto,
 kosho_jisshibi,
 koho.seimei as kohosha_seimei,
 kohosha_sql.name as kohosha_rank,
 kosho_seq as kaisu,
 case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as kohosha_apsource,
 shokai_sql.juryosha,
 shoki.ap_kakutoku as shokikosho_apkakutokusha_moto,
 case
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is null then shoki.mendan_tanto
    when shoki.ap_kakutoku is null and consts.name != "パートナー紹介" then shoki.mendan_tanto
    else shoki.ap_kakutoku end as shokikosho_apkakutokusha,
   
 case when shoki.partner_id is not null then shoki.partner_id
    when shoki.jinjibu_id is not null then shoki.jinjibu_id
    else null end as juryo_id,

 DATE_DIFF(kosho_jisshibi, LAG(kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY kosho_jisshibi), DAY) AS keikabi
from `r-group-bigdata.live_rhs.shokikoshos` as shoki
left join `r-group-bigdata.live_company.syain` syi1 on shoki.mendan_tanto = syi1.user_id
left join `r-group-bigdata.live_company.syain` syi2 on shoki.ap_kakutoku = syi2.user_id
left join
 (select id,seimei,kohosha_rank from `r-group-bigdata.live_rhs.kohoshas`) as koho ON shoki.kohosha_id = koho.id
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON koho.kohosha_rank = kohosha_sql.code
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as consts ON shoki.ap_source = consts.code
left join
 (select kohosha_id,juryosha from `r-group-bigdata.live_rhs.shokaijuryos`) as shokai_sql ON shoki.kohosha_id = shokai_sql.kohosha_id)

select *,
    LAG(kohosha_apsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_kohosha_apsource,
  from base)

select *,
   CASE WHEN kohosha_apsource != prev_kohosha_apsource THEN 1 ELSE 0
        END AS APS_change
  from base2),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

select
 id,
 kohosha_id,
 kohosha_seimei,
 kosho_setteibi,
 kosho_yoteibi,
 kosho_jisshibi,
 mendan_tanto,
 kohosha_rank,
 kaisu,
 kohosha_apsource,
 juryosha,
 shokikosho_apkakutokusha_moto,
 shokikosho_apkakutokusha,
 juryo_id,
 keikabi,
 prev_kohosha_apsource,
 APS_change,
 keikabi,
 case when kaisu = 1 then 1
      when APS_change = 1 then 1
      when keikabi > 90 then 2
      else 0 end as sai_flg,
 cal.quarter as shoki_jisshi_saiQ
from base3
left join cal on base3.kosho_jisshibi = cal.date
order by kosho_setteibi desc)

-- 再初期交渉実施
SELECT 
  "shoki_jisshi_sai" as type,
  concat("shoki_jisshi_sai",mendan_tanto) as key,
  mendan_tanto,
  shoki_jisshi_saiQ as shoki_jisshi_saiQ,
  COUNT(shoki_jisshi_saiQ) AS count_shoki_jisshi_saiQ
FROM ALLBASE
WHERE sai_flg = 2
GROUP BY mendan_tanto, shoki_jisshi_saiQ
ORDER BY shoki_jisshi_saiQ


"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
saikosho_data = client.query(sql).result().to_dataframe()


In [35]:
shoki_jisshi_sai = saikosho_data.copy()

shoki_jisshi_sai['join_key'] = shoki_jisshi_sai['mendan_tanto'] + shoki_jisshi_sai['shoki_jisshi_saiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
shoki_jisshi_sai = pd.merge(shoki_jisshi_sai, master2, on='join_key', how='inner')
shoki_jisshi_sai_kojin = shoki_jisshi_sai.pivot_table(index=["key","mendan_tanto"],columns="shoki_jisshi_saiQ",aggfunc="sum",values="count_shoki_jisshi_saiQ").fillna(0)

shoki_jisshi_sai["key2"] = shoki_jisshi_sai["type"] + shoki_jisshi_sai["レイヤー"]
shoki_jisshi_sai_layer = shoki_jisshi_sai.pivot_table(index=["key2","レイヤー"],columns="shoki_jisshi_saiQ",aggfunc="sum",values="count_shoki_jisshi_saiQ").fillna(0)

shoki_jisshi_sai = pd.concat([shoki_jisshi_sai_kojin,shoki_jisshi_sai_layer])

# 新規からの本交渉設定

In [36]:
sql = """
with ALLBASE as (
with HONS as (
with SHOKIS as (
with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_jisshibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs ON shk.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (
  SELECT 
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19
) consts ON shk.ap_source = consts.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name,
  case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as APsource,
  initial_contact_date,
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  shoki.kosho_seq,
  shoki.sai_flg,
  DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) as jisshibi_sa,
  cal1.quarter as hon_setteiQ,
  cal2.quarter as hon_jisshiQ
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
Left join cal cal1 on hon.kosho_setteibi = cal1.date
Left join cal cal2 on hon.kosho_jisshibi = cal2.date
where hon.kosho_seq = 1)

select *,
  case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HONS
order by anken_id)

-- 新規からの本交渉設定
SELECT 
  "hon_settei_shin" as type,
  concat("hon_settei_shin",kohosha_tanto) as key,
  kohosha_tanto,
  hon_setteiQ,
  COUNT(hon_setteiQ) AS count_hon_setteiQ
FROM ALLBASE
WHERE sai_flg2 = 1 or sai_flg2 = 0
GROUP BY kohosha_tanto, hon_setteiQ
ORDER BY hon_setteiQ
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()

In [37]:
hon_settei_shin = honkosho_data.copy()

hon_settei_shin['join_key'] = hon_settei_shin['kohosha_tanto'] + hon_settei_shin['hon_setteiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']

hon_settei_shin = pd.merge(hon_settei_shin, master2, on='join_key', how='inner')
hon_settei_shin_kojin = hon_settei_shin.pivot_table(index=["key","kohosha_tanto"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_setteiQ").fillna(0)


hon_settei_shin["key2"] = hon_settei_shin["type"] + hon_settei_shin["レイヤー"]
hon_settei_shin_layer = hon_settei_shin.pivot_table(index=["key2","レイヤー"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_setteiQ").fillna(0)

hon_settei_shin = pd.concat([hon_settei_shin_kojin,hon_settei_shin_layer])


# 再交渉からの設定

In [38]:
sql = """
with ALLBASE as (
with HONS as (
with SHOKIS as (
with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_jisshibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs ON shk.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (
  SELECT 
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19
) consts ON shk.ap_source = consts.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name,
  case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as APsource,
  initial_contact_date,
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  shoki.kosho_seq,
  shoki.sai_flg,
  DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) as jisshibi_sa,
  cal1.quarter as hon_setteiQ,
  cal2.quarter as hon_jisshiQ
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
Left join cal cal1 on hon.kosho_setteibi = cal1.date
Left join cal cal2 on hon.kosho_jisshibi = cal2.date
where hon.kosho_seq = 1)

select *,
  case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HONS
order by anken_id)

-- 新規からの本交渉設定
SELECT
  "hon_settei_sai" as type,
  concat("hon_settei_sai",kohosha_tanto) as key,
  kohosha_tanto,
  hon_setteiQ,
  COUNT(hon_setteiQ) AS count_hon_settei_saiQ
FROM ALLBASE
WHERE sai_flg2 = 2
GROUP BY kohosha_tanto, hon_setteiQ
ORDER BY hon_setteiQ
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()


In [39]:
hon_settei_sai = honkosho_data.copy()

hon_settei_sai['join_key'] = hon_settei_sai['kohosha_tanto'] + hon_settei_sai['hon_setteiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']

hon_settei_sai = pd.merge(hon_settei_sai, master2, on='join_key', how='inner')
hon_settei_sai_kojin = hon_settei_sai.pivot_table(index=["key","kohosha_tanto"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_settei_saiQ").fillna(0)


hon_settei_sai["key2"] = hon_settei_sai["type"] + hon_settei_sai["レイヤー"]
hon_settei_sai_layer = hon_settei_sai.pivot_table(index=["key2","レイヤー"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_settei_saiQ").fillna(0)

hon_settei_sai = pd.concat([hon_settei_sai_kojin,hon_settei_sai_layer])


# 新規からの本交渉（人）

In [40]:
sql = """
with ALLBASE as (
with HON_NINS2 as (
with HON_NINS as (
with SHOKIS as (
  with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_jisshibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs ON shk.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (
  SELECT 
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19
) consts ON shk.ap_source = consts.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name as company_name,
  case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as ap_source,
  initial_contact_date,     
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  shoki.kosho_seq,
  sai_flg,
  cal1.quarter as honnin_setteiQ,
  cal2.quarter as honnin_jisshiQ
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
left join cal cal1 on hon.kosho_setteibi = cal1.date
left join cal cal2 on hon.kosho_jisshibi = cal2.date
WHERE hon.kosho_seq = 1)

select distinct 
 kohosha_id,
--  tsr_code,
 ap_source,
 initial_contact_date,
 shoki_jisshibi,
 min(hon_setteibi) as hon_setteibi,
 kohosha_tanto,
 honnin_setteiQ,
 kosho_seq,
 sai_flg,
 DATE_DIFF(PARSE_DATE('%Y/%m/%d', min(hon_setteibi)), PARSE_DATE('%Y/%m/%d', shoki_jisshibi), DAY) as jisshibi_sa,
from HON_NINS
group by
 kohosha_id,
--  tsr_code,
 ap_source,
 initial_contact_date,
 shoki_jisshibi,
 kohosha_tanto,
 honnin_setteiQ,
 kosho_seq,
 sai_flg
order by hon_setteibi)

select *,
 case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HON_NINS2)

-- 新規からの本交渉設定(人)
SELECT 
  concat("honnin_settei_shin",kohosha_tanto) as key,
  kohosha_tanto,
  honnin_setteiQ,
  COUNT(honnin_setteiQ) AS count_honnin_setteiQ
FROM ALLBASE
WHERE sai_flg2 = 1 or sai_flg2 = 0
GROUP BY kohosha_tanto, honnin_setteiQ
ORDER BY honnin_setteiQ
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honnin_data = client.query(sql).result().to_dataframe()
honnin_settei_shin = honnin_data.copy()
honnin_settei_shin = honnin_settei_shin.pivot_table(index=["key","kohosha_tanto"],columns="honnin_setteiQ",aggfunc="sum",values="count_honnin_setteiQ").fillna(0)


# 再交渉からの本交渉（人）

In [41]:
sql = """
with ALLBASE as (
with HON_NINS2 as (
with HON_NINS as (
with SHOKIS as (
  with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_jisshibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs ON shk.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (
  SELECT 
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19
) consts ON shk.ap_source = consts.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name as company_name,
  case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as ap_source,
  initial_contact_date,     
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  shoki.kosho_seq,
  sai_flg,
  cal1.quarter as honnin_setteiQ,
  cal2.quarter as honnin_jisshiQ
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
left join cal cal1 on hon.kosho_setteibi = cal1.date
left join cal cal2 on hon.kosho_jisshibi = cal2.date
WHERE hon.kosho_seq = 1)

select distinct 
 kohosha_id,
--  tsr_code,
 ap_source,
 initial_contact_date,
 shoki_jisshibi,
 min(hon_setteibi) as hon_setteibi,
 kohosha_tanto,
 honnin_setteiQ,
 kosho_seq,
 sai_flg,
 DATE_DIFF(PARSE_DATE('%Y/%m/%d', min(hon_setteibi)), PARSE_DATE('%Y/%m/%d', shoki_jisshibi), DAY) as jisshibi_sa,
from HON_NINS
group by
 kohosha_id,
--  tsr_code,
 ap_source,
 initial_contact_date,
 shoki_jisshibi,
 kohosha_tanto,
 honnin_setteiQ,
 kosho_seq,
 sai_flg
order by hon_setteibi)

select *,
 case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HON_NINS2)

-- 新規からの本交渉設定(人)
SELECT 
  concat("honnin_settei_sai",kohosha_tanto) as key,
  kohosha_tanto,
  honnin_setteiQ,
  COUNT(honnin_setteiQ) AS count_honnin_settei_saiQ
FROM ALLBASE
WHERE sai_flg2 = 2
GROUP BY kohosha_tanto, honnin_setteiQ
ORDER BY honnin_setteiQ
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honnin_data = client.query(sql).result().to_dataframe()
honnin_settei_sai = honnin_data.copy()
honnin_settei_sai = honnin_settei_sai.pivot_table(index=["key","kohosha_tanto"],columns="honnin_setteiQ",aggfunc="sum",values="count_honnin_settei_saiQ").fillna(0)


# 企業設定数

In [42]:
sql = """
with ALLBASE as (
with HONS as (
with SHOKIS as (
with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_jisshibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs ON shk.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (
  SELECT 
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19
) consts ON shk.ap_source = consts.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
),

cal as (
  SELECT 
   yyyymmdd as date, 
   concat(ki,"-",q,"Q") as quarter
  FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
  order by yyyymmdd)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name,
  case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as APsource,
  initial_contact_date,
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  shoki.kosho_seq,
  shoki.sai_flg,
  DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) as jisshibi_sa,
  cal1.quarter as hon_setteiQ,
  cal2.quarter as hon_jisshiQ
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
Left join cal cal1 on hon.kosho_setteibi = cal1.date
Left join cal cal2 on hon.kosho_jisshibi = cal2.date
where hon.kosho_seq = 1)

select *,
  case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HONS
order by anken_id)

-- 新規からの本交渉設定
SELECT 
  "kigyo_settei_shin" as type,
  concat("kigyo_settei_shin",kigyo_tanto) as key,
  kigyo_tanto,
  hon_setteiQ,
  COUNT(hon_setteiQ) AS count_hon_setteiQ
FROM ALLBASE
# WHERE sai_flg2 = 1 or sai_flg2 = 0
GROUP BY kigyo_tanto, hon_setteiQ
ORDER BY hon_setteiQ
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()


In [43]:
kigyo_settei_shin = honkosho_data.copy()

kigyo_settei_shin['join_key'] = kigyo_settei_shin['kigyo_tanto'] + kigyo_settei_shin['hon_setteiQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
kigyo_settei_shin = pd.merge(kigyo_settei_shin, master2, on='join_key', how='inner')
kigyo_settei_shin_kojin = kigyo_settei_shin.pivot_table(index=["key","kigyo_tanto"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_setteiQ").fillna(0)


kigyo_settei_shin["key2"] = kigyo_settei_shin["type"] + kigyo_settei_shin["レイヤー"]
kigyo_settei_shin_layer = kigyo_settei_shin.pivot_table(index=["key2","レイヤー"],columns="hon_setteiQ",aggfunc="sum",values="count_hon_setteiQ").fillna(0)

kigyo_settei_shin = pd.concat([kigyo_settei_shin_kojin,kigyo_settei_shin_layer])


# 企業設定社数

In [44]:
sql = """
with ALLBASE as (
with rownumber_kami2 as (
with rownumber_kami as (
with jikeirestufubi_jokyo as (
with hon_sha as (
 select
  kigyo.tsr_code, 
  sya.sei_plus as kigyo_tanto, 
  min(hon.kosho_setteibi) kosho_setteibi,
  cal.Q as hon_shaQ
 from `r-group-bigdata.live_rhs.honkoshos` hon 
 left join `r-group-bigdata.live_rhs.ankens` an on an.id = hon.anken_id
 left join `r-group-bigdata.live_rhs.kigyos` kigyo on an.kigyo_id = kigyo.id
 left join `r-group-bigdata.live_rhs.ronzan_members` mem on an.kigyo_tanto = mem.userid
 left join `r-group-bigdata.live_company.syain` sya on an.kigyo_tanto = sya.user_id
 LEFT JOIN (SELECT yyyymmdd as date,
            concat(ki,"-",q,"Q") as Q
            FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`) cal on hon.kosho_setteibi = cal.date
 where (hon.kosho_seq = 1 or hon.kosho_seq_extra = 1)
 and mem.id is not null
--  and hon.kosho_setteibi >= "2023-10-03"
--  and hon.kosho_setteibi <= "2023-11-08"
 group by
  sya.sei_plus, 
  an.kigyo_id,
  kigyo.tsr_code,
  cal.Q),

sales as (
with ranked as(
 SELECT 
    ap.id as ap_id,
    ap.tsr_code,
    ap.appoint_visit_plan_date,
    shuho.name as shuho,
    case when consts.name = "スカウト現S" then "スカウト現S"
         when consts.name like "%RZ現S%" then "ロンザン現元S"
         when consts.name like "%RZ元S%" then "ロンザン現元S"
         else "それ以外" end as apsource,
    ROUND(tokikessan_uriagedaka/100000,0)as URIAGE,
    ful.price_explanation3 as price,
    cal.q as salesQ,
    ROW_NUMBER() OVER (PARTITION BY ap.company_name, cal.q ORDER BY ap.appoint_visit_plan_date asc) as rn
  FROM `r-group-bigdata.live_rhs.sales_appoints` ap
  left join `r-group-bigdata.live_rhs.sales_appoint_fulfills` ful on ap.id = ful.id
  left join `r-group-bigdata.tsr.company_info` tsr on ap.tsr_code = tsr.tsr_code
  left join (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 5500) shuho on cast(ap.shuho as int) = shuho.code
  LEFT JOIN (SELECT code, name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 110) consts on cast(ap.appoint_source as int) = consts.code
  LEFT JOIN (SELECT yyyymmdd as date,
            concat(ki,"-",q,"Q") as Q
            FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`) cal on ap.appoint_visit_plan_date = cal.date)

select *
from ranked 
where rn = 1
)  

select
 ap_id,
 hon_sha.tsr_code,
 URIAGE,
 kosho_setteibi,
 kigyo_tanto,
 appoint_visit_plan_date,
 shuho,
 apsource,
 hon_shaQ,
 case when sales.appoint_visit_plan_date is null then 1
      when sales.appoint_visit_plan_date <= hon_sha.kosho_setteibi then 1
      else 0 end jikeiretsu,
 case when price = 100 then "説明無し"
       when price = 200 then "58%+9%"
       when price = 300 then "62%+12%"
       when price = 400 then "65%+12%"
       when price = 500 then "67%+14%"
       when price = 600 then "固定報酬165万円＋69％＋16％"
       when price = 650 then "固定報酬120万円＋69％＋16％"
       when price = 700 then "固定報酬60万円＋69％＋16％"
       when price = 800 then "半常勤プラン"
       when price = 900 then "その他"
       else "-" end as price
from hon_sha
left join sales on hon_sha.tsr_code = sales.tsr_code)

select 
 *
from jikeirestufubi_jokyo
where jikeiretsu = 1
)

select 
 *,
 row_number () over (partition by tsr_code,hon_shaQ order by appoint_visit_plan_date desc) as row_num
from rownumber_kami)

select 
 *
from rownumber_kami2
where row_num = 1)

select 
  "kigyo_settei_sha" as type,
  concat("kigyo_settei_sha",kigyo_tanto) as key,
  kigyo_tanto,
  hon_shaQ,
  COUNT(hon_shaQ) AS count_hon_sha_setteiQ
FROM ALLBASE
# WHERE sai_flg2 = 1 or sai_flg2 = 0
GROUP BY kigyo_tanto, hon_shaQ
ORDER BY hon_shaQ
 
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_sha_data = client.query(sql).result().to_dataframe()


In [45]:
kigyo_settei_sha = honkosho_sha_data.copy()
kigyo_settei_sha['join_key'] = kigyo_settei_sha['kigyo_tanto'] + kigyo_settei_sha['hon_shaQ']
master2['join_key'] = master2['sei_plus'] + master2['Q']
kigyo_settei_sha = pd.merge(kigyo_settei_sha, master2, on='join_key', how='inner')

kigyo_settei_sha_kojin = kigyo_settei_sha.pivot_table(index=["key","kigyo_tanto"],columns="hon_shaQ",aggfunc="sum",values="count_hon_sha_setteiQ").fillna(0)


kigyo_settei_sha["key2"] = kigyo_settei_sha["type"] + kigyo_settei_sha["レイヤー"]
kigyo_settei_sha_layer = kigyo_settei_sha.pivot_table(index=["key2","レイヤー"],columns="hon_shaQ",aggfunc="sum",values="count_hon_sha_setteiQ").fillna(0)

kigyo_settei_sha = pd.concat([kigyo_settei_sha_kojin,kigyo_settei_sha_layer])


<!-- # 再交渉からの企業設定 -->

In [46]:
shoki_hon_data = pd.concat([shoki_settei_shin,shoki_jisshi_shin,
                            shoki_settei_sai,shoki_jisshi_sai,
                            hon_settei_shin,hon_settei_sai,
                            honnin_settei_shin,honnin_settei_sai,
                            kigyo_settei_shin,
                            kigyo_settei_sha
                            ], axis=0).reset_index()


In [47]:
shoki_hon_final = shoki_hon_data.fillna(0)

In [48]:
final = [shoki_hon_final.columns.tolist()] + shoki_hon_final.values.tolist()


In [49]:
# スプレッドシートに貼り付ける
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1Fq7rbtt6--bkl-laypFFPqYDuTvy_Vps4p30j7FoSS4'
Sheet_NAME = 'data2!D'
Sheet_row = "1"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,final,service)


# 成約

In [50]:

col_name = ["案件id\n（RZ）","計上月","期","月","案件\nNo\n（SC）","入力者","計上日","受注日","クライアント正式名称","候補者",
            "売上種別（商品内容）","差分\n（該当場合のみ）","紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）","基準年収","報酬率",
            "完保\n成約\n割振比","グループ企画料","サービス\n引当\n係数","キャンセル\n引当\n係数","営業売上合計","所属課",
            "氏名","割合","受注額","査定用\n売上","顧客支持ポイント","引き継ぎP","担当","担当\n押印","備考①","備考②","備考③","シニアスカウト\n（ヨミ表と一致）",
            "内定数フラグ","企業アポソース","特殊フラグ","提示/前年度","基準年収.1","報酬率.1","d","d.1","アポソース"]

path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

#過去のポイントデータ（こちらは積み上げ式にしているのでExcelファイルからもってくる　※RPAとBTのクロスセル分は除いた状態にて積み上げ済）
yomi_old_betu = pd.read_excel(path1 + "\yojitu\RZポイント表過去データ（22-3Qまで）最新.xlsx",usecols=[0,1,2,3,4])
yomi_old_betu = yomi_old_betu.rename(columns={"両手案件フラグ":"旧両手","アポソース丸め":"旧アポ",
                                                "候補者アポソース":"旧候補者アポ","候補者id":"旧候id"})

yomi_old = pd.read_excel(path1 + "\yojitu\RZポイント表過去データ（22-3Qまで）最新.xlsx",usecols  =[5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47])[col_name]
yomi_old = yomi_old.rename(columns={"営業売上\n（営業ポイント）.1":"営業売上\n（営業ポイント）"})
#yomi_old.to_excel('22-3Qデータ.xlsx')


In [51]:
#今Qのポイントデータは最新のヨミ表からとってくる
yomi_nowQ = pd.read_excel(r"\\172.16.0.232\CoffeeCrazy\管理グループ\管理部\ポイント割り振り表\【入力用】ポイント表\27期\27-3\ロンザン統合版ポイント表\【27-3Q】入力用顧客支持ポイント表（ロンザン）.xlsm",header=20,usecols = (range(0, 43)))
yomi_nowQ = yomi_nowQ[:-1]

#RPA事業部のクロスセルは除く(顧問名が「塩澤昌紘」はRPA事業部のクロスセル案件)
yomi_nowQ = yomi_nowQ[~yomi_nowQ['候補者'].isin(['塩澤昌紘'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['RPAコンサル'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['ビジネスタンク'])]
yomi_nowQ.to_excel('yomi_nowQ.xlsx')

yomi_nowQ.head(2)

,案件id\n（RZ）,計上月,期,月,案件\nNo\n（SC）,入力者,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊顧客支持ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,引き継ぎP,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト\n（ヨミ表と一致）,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,ロンザン氏名\n表記揺れチェック
0,NaN,4月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,5月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
#過去計上済のヨミ表情報と、今Qのヨミ表とをくっつける
yomi = pd.concat([yomi_old,yomi_nowQ],axis=0,ignore_index=True)

In [53]:
#ポイントデータを結合する
yomi_data =yomi.copy()
yomi_data.index = yomi_data.index + 1
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.set_index("index")

#上からナンバーを割り振る
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.rename(columns={"index":"発番"})
#yomi_data.to_excel('総合ポイント.xlsx')

yomi_data.head(1)

,発番,案件id\n（RZ）,計上月,期,月,案件\nNo\n（SC）,入力者,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,引き継ぎP,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト\n（ヨミ表と一致）,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,紹介引当/PM引当\n特殊顧客支持ポイント\n（該当場合のみ）,ロンザン氏名\n表記揺れチェック
0,1,NaN,4月,19.0,4.0,8606,石田,2016-04-15 00:00:00,2016-03-31 00:00:00,株式会社フジダン,白川 正明 氏,スカウト報酬（通常）,NaN,NaN,630.0,0.55,1.0,NaN,NaN,NaN,NaN,ロンザン,宮崎佳,0.4,138.6,97.02,82.467,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
yomi_data["案件id\n（RZ）"]=pd.to_numeric(yomi_data["案件id\n（RZ）"],errors='coerce')
yomi_data["案件id\n（RZ）"]=yomi_data["案件id\n（RZ）"].fillna(0.0).astype(int)
yomi_data["案件id\n（RZ）"]=yomi_data["案件id\n（RZ）"].astype(int)


In [55]:
# hon_df
yomi_data


,発番,案件id\n（RZ）,計上月,期,月,案件\nNo\n（SC）,入力者,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,引き継ぎP,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト\n（ヨミ表と一致）,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,紹介引当/PM引当\n特殊顧客支持ポイント\n（該当場合のみ）,ロンザン氏名\n表記揺れチェック
0,1,0,4月,19.0,4.0,8606,石田,2016-04-15 00:00:00,2016-03-31 00:00:00,株式会社フジダン,白川 正明 氏,スカウト報酬（通常）,NaN,NaN,630.0,0.55,1.0,NaN,NaN,NaN,NaN,ロンザン,宮崎佳,0.4,138.6,97.02,82.467,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,0,4月,19.0,4.0,NaN,石田,2016-04-15 00:00:00,2016-04-15 00:00:00,株式会社林間,岩本 昌和 氏,スカウト報酬（通常）,NaN,NaN,740.0,0.58,1.0,NaN,NaN,NaN,NaN,ロンザン,宮崎佳,0.4,171.68,120.176,102.1496,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,0,4月,19.0,4.0,NaN,石田,2016-04-21 00:00:00,2016-04-21 00:00:00,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,NaN,NaN,1110.0,0.58,1.0,643.8,NaN,NaN,NaN,シニア引当,サービス引当,0.1,64.38,64.38,64.38,NaN,NaN,NaN,本採用内定確認書は平成28年1月20日付で締結した人材コンサルティングサービスに関する契約書...,1,B,●,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,0,4月,19.0,4.0,NaN,石田,2016-04-21 00:00:00,2016-04-21 00:00:00,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,NaN,NaN,1110.0,0.58,1.0,643.8,NaN,NaN,NaN,シニア引当,CXL引当,0.2,128.76,128.76,128.76,NaN,NaN,NaN,NaN,NaN,NaN,●,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,0,4月,19.0,4.0,NaN,石田,2016-04-21 00:00:00,2016-04-21 00:00:00,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,NaN,NaN,1110.0,0.58,1.0,643.8,NaN,NaN,NaN,ロンザン,長崎文,0.2975,191.5305,191.5305,191.5305,NaN,候補者担当,NaN,NaN,NaN,NaN,●,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54188,54189,0,6月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54189,54190,0,6月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54190,54191,0,6月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54191,54192,0,6月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:

#設定元データと最新ポイントファイルを結合 (アポソース等を紐付けるため)
honkosho_settei_data = hon_df[['honkosho_id','anken_id','kohosha_id','hon_setteibi','kohosha_tanto','APsource','sai_flg2']].copy()
honkosho_settei_data = honkosho_settei_data.rename(columns={"anken_id":"案件id\n（RZ）"})
honkosho_settei_data= honkosho_settei_data.drop_duplicates(subset=["案件id\n（RZ）"],keep='first')
    

yomi_data = pd.merge(yomi_data,honkosho_settei_data,on = "案件id\n（RZ）",how="left")

NameError: name 'hon_df' is not defined

In [ ]:
#ヨミ表データと過去のヨミ表データを紐付ける（候補者アポソース取得のため）
yomi_data = pd.merge(yomi_data,yomi_old_betu,on = "発番",how="left")

#企業担当フラグ（この成約の企業担当にフラグを付ける）
yomi_data["企業担当フラグ"] = yomi_data.apply(lambda x : 1 if x["担当"] in ("面談担当","面談担当①","面談担当・候補者担当") else 0 ,axis = 1)

#候補者IDが紐づかない受注については、過去使った候補者IDを紐付ける
yomi_data["kohosha_id"] = yomi_data["kohosha_id"].fillna(0.0)
yomi_data["候補者id"] = yomi_data.apply(lambda x : x["旧候id"] if x["kohosha_id"] == 0 else x["kohosha_id"] ,axis = 1)



KeyError: 'kohosha_id'

In [ ]:
#それぞれの日付に同営業日、Qなどの情報を付加する。
Date_df = DATE_DATA.copy()
Date_df = Date_df.rename(columns={"日付":"計上日"})

Date_df['計上日'] = pd.to_datetime(Date_df['計上日'])
yomi_data['計上日'] = pd.to_datetime(yomi_data['計上日'])

yomi_data = pd.merge(yomi_data,Date_df,on = ("計上日"),how = "left")



In [ ]:
#それぞれのデータにPhaseを分ける
yomi_data.insert(0, 'Pahse', 'seiyaku'+yomi_data["kohosha_tanto"])


In [ ]:
#Phaseと面談担当でPivotする
yomi_all = yomi_data.pivot_table(index=["Pahse","kohosha_tanto"],columns="Q",aggfunc="count",values="sai_flg2").fillna(0)


In [ ]:
honnin_all = pd.concat([honnin_all_settei, honnin_settei_shin, honnin_settei_sai], axis=0).reset_index()
shokihon_all = pd.concat([shoki_sai,hon_all], axis=0).reset_index()

final_data = pd.concat([shokihon_all,yomi_all])

final_data = final_data.fillna(0)


In [ ]:
final_data = [final_data.columns.tolist()] + final_data.values.tolist()


In [ ]:
#日付データを取る
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1183f9zbvJR7y2-Lbp1DnyI5aLc6xD6nIcSiEWBG4lfA'
Sheet_NAME = 'data!A'
Sheet_row = "1"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,final_data,service)


TypeError: Object of type DataFrame is not JSON serializable